# Lesson 15 — Exercises

Run `setup.sql` first, then work through `lesson.ipynb`. Solutions are in `exercises-solutions.ipynb`.

Work the exercises in order. Exercises 2 and 4 mention the actual execution plan (Ctrl+M) — that graphical view is an SSMS feature; run the queries in ADS to see results and switch to SSMS to compare Estimated vs Actual rows.

In [ ]:
USE AdventureWorks2022;
GO

## Exercise 1

Use `DBCC SHOW_STATISTICS` on `IX_lesson15_SOH_OrderDate`. From the histogram, pick a RANGE_HI_KEY date and report its EQ_ROWS (estimated rows equal to that key).

In [ ]:
DBCC SHOW_STATISTICS ('lesson15.SalesOrderHeader', 'IX_lesson15_SOH_OrderDate');

**Why:** the third result set lists RANGE_HI_KEY steps; EQ_ROWS is the estimate for "= key". Read any RANGE_HI_KEY row from the histogram and note its EQ_ROWS column. (Exact value depends on AdventureWorks data; e.g. a 2014 date with its EQ_ROWS estimate.)

## Exercise 2

Run the query below with the ACTUAL plan on (Ctrl+M in SSMS). Report the Estimated vs Actual number of rows on the seek/scan operator.

In [ ]:
SELECT SalesOrderID, TotalDue FROM lesson15.SalesOrderHeader WHERE OrderDate >= '2014-01-01';

**Why:** with FULLSCAN stats the optimizer's estimate should be close to actual. In the actual plan, the operator tooltip shows "Estimated Number of Rows" vs "Actual Number of Rows" — record both; they should be close on fresh statistics.

## Exercise 3

Find the cached plan(s) for any query touching `lesson15.SalesOrderHeader` and report the highest usecounts (reuse count).

In [ ]:
SELECT TOP 5 cp.usecounts, cp.objtype, SUBSTRING(st.text, 1, 120) AS QueryTextStart
FROM sys.dm_exec_cached_plans AS cp
CROSS APPLY sys.dm_exec_sql_text(cp.plan_handle) AS st
WHERE st.text LIKE '%lesson15.SalesOrderHeader%'
  AND st.text NOT LIKE '%dm_exec_cached_plans%'
ORDER BY cp.usecounts DESC;

**Why:** join cached plans to their SQL text and sort by usecounts.

## Exercise 4

Deliberately make statistics stale: reassign a large batch of rows to CustomerID = 11001 via UPDATE (do NOT update stats yet). Then run a query filtering CustomerID = 11001 with the actual plan on and observe the estimate is now too low. Finally fix it with `UPDATE STATISTICS` and re-run.

In [ ]:
UPDATE TOP (10000) lesson15.SalesOrderHeader
SET CustomerID = 11001
WHERE CustomerID <> 11001;
GO
-- Run with actual plan on: Estimated rows << Actual rows for this customer.
SELECT SalesOrderID, OrderDate FROM lesson15.SalesOrderHeader WHERE CustomerID = 11001;
-- Fix: refresh stats so the estimate matches again.
UPDATE STATISTICS lesson15.SalesOrderHeader WITH FULLSCAN;
SELECT SalesOrderID, OrderDate FROM lesson15.SalesOrderHeader WHERE CustomerID = 11001;

**Why:** reassign many rows to one customer via UPDATE (no identity/PK conflict); the estimate lags reality until stats refresh.

## Exercise 5

In a comment, explain the four phases of the query lifecycle and name the phase where statistics are used.

The four phases of the query lifecycle:

1) Parse  - check syntax, build a parse tree.
2) Bind (algebrize) - resolve object/column names, types; build a logical tree.
3) Optimize - the cost-based optimizer uses STATISTICS to estimate cardinality and pick a plan.
4) Execute - the chosen plan runs; the plan is cached for reuse.

Statistics are used in the OPTIMIZE phase, driving cardinality estimates and plan choice.